In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import warnings

# Suppress warnings for a cleaner intern-level presentation
warnings.filterwarnings('ignore')

# Set visual style
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Assuming the data is in the ../data/ folder relative to this notebook
try:
    sentiment_df = pd.read_csv('../data/bitcoin_sentiment.csv')
    trader_df = pd.read_csv('../data/hyperliquid_data.csv')
    
    print("Datasets loaded successfully.")
    print(f"Sentiment Data Shape: {sentiment_df.shape}")
    print(f"Trader Data Shape: {trader_df.shape}")
except FileNotFoundError:
    print("Error: Check if CSV files exist in the '../data/' directory.")

In [ ]:
# Convert sentiment dates
sentiment_df['Date'] = pd.to_datetime(sentiment_df['Date']).dt.date

# Convert Hyperliquid 'time' (ms) to normalized daily dates
trader_df['time'] = pd.to_datetime(trader_df['time'], unit='ms')
trader_df['Date'] = trader_df['time'].dt.date

# Merging datasets on Date to explore the relationship
df = pd.merge(trader_df, sentiment_df, on='Date', how='inner')

print("Data cleaning complete. Sample of merged data:")
df.head()

In [ ]:
# Exploring PnL and Leverage across different sentiment categories
sentiment_stats = df.groupby('Classification').agg({
    'closedPnL': ['mean', 'median', 'std'],
    'leverage': 'mean',
    'side': 'count'
}).reset_index()

print("Summary Statistics by Sentiment Phase:")
print(sentiment_stats)

In [ ]:
# Visualization 1: Distribution of PnL across Sentiment Phases
plt.figure(figsize=(14, 7))
sns.violinplot(data=df, x='Classification', y='closedPnL', inner="quart", palette="muted")
plt.title("Trader PnL Distribution vs. Market Sentiment (Fear/Greed)")
plt.axhline(0, color='red', linestyle='--', alpha=0.5)
plt.show()

# Visualization 2: Correlation between Leverage and Sentiment
plt.figure(figsize=(10, 5))
sns.barplot(data=df, x='Classification', y='leverage', ci=None, palette="magma")
plt.title("Average Leverage Usage per Sentiment Phase")
plt.show()

In [ ]:
# Identifying the 'Best' phase for traders in this dataset
best_phase = df.groupby('Classification')['closedPnL'].mean().idxmax()
worst_phase = df.groupby('Classification')['closedPnL'].mean().idxmin()

print(f"--- PRELIMINARY INSIGHTS ---")
print(f"1. Traders were most profitable during '{best_phase}' phases.")
print(f"2. The lowest average performance occurred during '{worst_phase}' phases.")
print(f"3. Further analysis is required to see if high leverage correlates with losses during Fear.")